# 🔢 Count Models — Poisson, Negative Binomial & ZINB
**ISLP Chapter 4 (Extended) · Pattern Recognition for the Rest of Us**

> When your outcome is a count — doctor visits, insurance claims, website clicks, defects — linear regression is the wrong tool. Count models respect the discrete, non-negative, right-skewed nature of count data.

### What you'll learn
- Why OLS fails for count outcomes (with real evidence)
- **Poisson regression** — log-link, IRR interpretation
- **Negative Binomial** — when counts are overdispersed
- **ZINB** — when there are too many zeros
- Model diagnostics and choosing between models

### Dataset
**RAND Health Insurance Experiment** — 20,190 real patients, outpatient doctor visits.  
Built into statsmodels: no download, no URLs, always works.

### Time: ~70 minutes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
import statsmodels.datasets as smd
from statsmodels.discrete.count_model import ZeroInflatedNegativeBinomialP
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#f8f9fa',
    'axes.grid': True, 'grid.alpha': 0.4,
    'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 11
})
print("\u2713 All packages loaded")

In [ ]:
# RAND Health Insurance Experiment — built into statsmodels, no download needed
randhie = smd.randhie.load_pandas().data.copy()

print("=== RAND Health Insurance Experiment ===")
print(f"n = {len(randhie):,} individuals")
print("\nVariables:")
print("  mdvis   = # outpatient doctor visits  (COUNT OUTCOME)")
print("  lncoins = log(coinsurance+1)  — higher = more out-of-pocket cost")
print("  idp     = individual deductible plan (1=yes)")
print("  physlm  = physical limitation (1=yes)")
print("  disea   = number of chronic diseases")
print("  hlthg/f/p = self-rated health: good / fair / poor  (omitted = excellent)")
print(f"\nOutcome: mdvis (doctor visits)")
print(f"  Mean:         {randhie['mdvis'].mean():.2f}")
print(f"  Variance:     {randhie['mdvis'].var():.2f}")
print(f"  Var/Mean:     {randhie['mdvis'].var()/randhie['mdvis'].mean():.2f}  (>>1 = overdispersed)")
print(f"  Zero rate:    {(randhie['mdvis']==0).mean():.1%}")
print(f"  Max visits:   {randhie['mdvis'].max()}")
randhie.head()

## 📐 Part 1 — Why Linear Regression Fails for Count Data

OLS violates four properties of count data:
1. **Non-negative** — OLS predicts negative counts (impossible)
2. **Integer-valued** — OLS predicts continuous values
3. **Heteroscedastic** — variance grows with the mean (Var ≠ constant)
4. **Multiplicative effects** — predictors multiply expected counts, not add

**Fix:** model log(μ) as a linear function → predictions always positive, effects multiplicative.
```
log(E[Y|X]) = β₀ + β₁X₁ + β₂X₂ + ...
```

In [ ]:
# Why OLS fails on real count data
from sklearn.linear_model import LinearRegression

features = ['lncoins', 'idp', 'physlm', 'disea', 'hlthg', 'hlthf', 'hlthp']
X_ols = randhie[features].values
y     = randhie['mdvis'].values

ols   = LinearRegression().fit(X_ols, y)
y_hat = ols.predict(X_ols)
neg_pct = (y_hat < 0).mean() * 100
actual_ratio = randhie['mdvis'].var() / randhie['mdvis'].mean()

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Plot 1: true distribution
ax = axes[0]
ax.hist(randhie['mdvis'], bins=range(0, 40), color='#1e5fa8', edgecolor='white', alpha=0.85)
ax.set_title('Distribution of Doctor Visits\n(RAND data — n=20,190 real patients)')
ax.set_xlabel('# Outpatient Visits')
ax.set_ylabel('Count of Patients')
ax.axvline(randhie['mdvis'].mean(), color='#e85d20', lw=2,
           label=f"Mean = {randhie['mdvis'].mean():.1f}")
ax.legend()

# Plot 2: OLS predicts negatives
ax = axes[1]
ax.scatter(y_hat, y, alpha=0.07, s=8, color='#e85d20')
ax.axvline(0, color='#c0392b', lw=2.5, ls='--',
           label=f'{neg_pct:.1f}% predictions < 0 (impossible!)')
ax.set_xlabel('OLS Predicted Visits')
ax.set_ylabel('Actual Visits')
ax.set_title(f'OLS predicts {neg_pct:.1f}% negative values\n(impossible for a count outcome)')
ax.legend(fontsize=9)

# Plot 3: variance grows with mean
ax = axes[2]
bins_q = pd.qcut(randhie['mdvis'], q=15, duplicates='drop')
grp = randhie.groupby(bins_q, observed=True)['mdvis'].agg(['mean', 'var']).dropna()
ax.scatter(grp['mean'], grp['var'], color='#1a7a45', s=70, zorder=3,
           label='Observed (quantile bins)')
x_line = np.linspace(0, grp['mean'].max(), 100)
ax.plot(x_line, x_line,              color='#1e5fa8', lw=2, ls='--', label='Poisson: Var=Mean')
ax.plot(x_line, x_line + x_line**2, color='#e85d20', lw=2, ls='--', label='NB: Var=Mean+Mean^2')
ax.set_xlabel('Bin Mean')
ax.set_ylabel('Bin Variance')
ax.set_title(f'Variance >> Mean in real data\n(Var/Mean = {actual_ratio:.1f} — strongly overdispersed)')
ax.legend(fontsize=9)

plt.suptitle('Why OLS Fails for Count Data — RAND Health Insurance Data', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print(f"\n\u2714 Real-world evidence:")
print(f"   OLS predicts {neg_pct:.1f}% negative values — impossible for visit counts")
print(f"   Var/Mean = {actual_ratio:.1f} — Poisson assumes 1.0, real data is 7x overdispersed")
print(f"   31.2% zero visits — excess zeros to account for")

## 📊 Part 2 — Poisson Regression

**Incidence Rate Ratio (IRR) = exp(β):**
- IRR = 1.35 → predictor increases expected count by 35%
- IRR = 0.80 → predictor decreases expected count by 20%
- IRR = 1.00 → no effect

In [ ]:
# Fit Poisson regression
pois_model = smf.poisson(
    'mdvis ~ lncoins + idp + physlm + disea + hlthg + hlthf + hlthp',
    data=randhie
).fit(disp=False)

print(pois_model.summary())

In [ ]:
# Extract IRRs with 95% confidence intervals
coefs = pois_model.params.drop('Intercept')
irr   = np.exp(coefs)
ci    = np.exp(pois_model.conf_int().drop('Intercept'))
pvals = pois_model.pvalues.drop('Intercept')

irr_df = pd.DataFrame({
    'Coef (beta)': coefs.round(4),
    'IRR':         irr.round(4),
    'CI Low':      ci[0].round(4),
    'CI High':     ci[1].round(4),
    'p-value':     pvals.round(4),
    'Sig':         pvals < 0.05
})
print("=== Poisson IRR Table ===\n")
print(irr_df.to_string())

print("\n\u2714 Plain-English interpretation:")
for name, row in irr_df.iterrows():
    if row['Sig']:
        direction = 'increases' if row['IRR'] > 1 else 'decreases'
        pct = abs(row['IRR'] - 1) * 100
        print(f"  {name:12s}: {direction} expected visits by {pct:.0f}%  (p={row['p-value']:.4f})")

In [ ]:
# IRR forest plot
fig, ax = plt.subplots(figsize=(9, 5))

colors = ['#1a7a45' if v > 1 else '#e85d20' for v in irr_df['IRR']]
ax.barh(list(irr_df.index), irr_df['IRR'] - 1, left=1,
        color=colors, alpha=0.75, edgecolor='white', height=0.55)
ax.errorbar(irr_df['IRR'], range(len(irr_df)),
            xerr=[irr_df['IRR'] - irr_df['CI Low'],
                  irr_df['CI High'] - irr_df['IRR']],
            fmt='none', color='#1a1d23', capsize=5, lw=2)
ax.axvline(1, color='black', lw=2, ls='--', label='IRR=1 (no effect)')
ax.set_xlabel('Incidence Rate Ratio (IRR)')
ax.set_title('Poisson Regression: Effect of Each Predictor on Doctor Visits\n'
             'Green = increases visits, Orange = decreases visits')
ax.legend()
plt.tight_layout()
plt.show()

disea_irr = np.exp(pois_model.params['disea'])
coins_irr = np.exp(pois_model.params['lncoins'])
print(f"\n\u2714 disea IRR = {disea_irr:.3f}: each extra chronic disease multiplies visits by {disea_irr:.3f}")
print(f"\u2714 lncoins IRR = {coins_irr:.3f}: higher cost-sharing reduces visits")

## ⚠️ Part 3 — Overdispersion: Poisson Is Not Enough

The RAND data has **Var/Mean ≈ 7** — far exceeding Poisson's assumption of 1.0.

Ignoring overdispersion → **standard errors too small → false discoveries**.

**Negative Binomial** adds dispersion parameter α:  `Var(Y) = μ + α·μ²`

In [ ]:
# Test for overdispersion and fit Negative Binomial
pearson_resid = pois_model.resid_pearson
dispersion    = (pearson_resid**2).sum() / pois_model.df_resid

print("=== Overdispersion Diagnostics ===")
print(f"Pearson chi-squared / df = {dispersion:.2f}")
print("  Poisson assumes this equals 1.0")
print(f"  Our value is {dispersion:.1f}x the Poisson expectation -> strongly overdispersed")
print("  This means Poisson SEs are too small -> some 'significant' results are false")

# Fit Negative Binomial
nb_model = smf.negativebinomial(
    'mdvis ~ lncoins + idp + physlm + disea + hlthg + hlthf + hlthp',
    data=randhie
).fit(disp=False)

print("\n" + nb_model.summary().as_text())
print(f"\nEstimated alpha (dispersion): {nb_model.params['alpha']:.4f}")

In [ ]:
# Compare standard errors: Poisson vs Negative Binomial
pois_se = pois_model.bse.drop('Intercept')
nb_se   = nb_model.bse.drop(['Intercept', 'alpha'])

se_df = pd.DataFrame({
    'Poisson SE':    pois_se.round(5),
    'NegBin SE':     nb_se.round(5),
    'Ratio NB/Pois': (nb_se / pois_se).round(3)
})
print("=== Standard Errors: Poisson vs Negative Binomial ===\n")
print(se_df.to_string())
avg_ratio = (nb_se / pois_se).mean()
print(f"\n\u2714 NB SEs are {avg_ratio:.1f}x larger on average")
print("  Poisson was overconfident — it underestimated uncertainty due to overdispersion")

print(f"\n=== AIC Comparison (lower = better) ===")
print(f"  Poisson:          {pois_model.aic:>12,.1f}")
print(f"  Negative Binomial:{nb_model.aic:>12,.1f}  {'<-- much better!' if nb_model.aic < pois_model.aic else ''}")
print(f"  Delta AIC:        {pois_model.aic - nb_model.aic:>12,.0f}")

## 0️⃣ Part 4 — Zero-Inflated Models

**31.2% of patients have zero visits.** Even NB underpredicts this.

**ZINB** = mixture of:
- A **logistic model** for P(structural zero) — people who would never visit regardless
- A **Negative Binomial** for actual visit counts

In [ ]:
# Diagnose zero inflation
from scipy.stats import poisson as sp_poisson

mu_pois  = pois_model.predict()
mu_nb    = nb_model.predict()
alpha_nb = nb_model.params['alpha']
r        = 1.0 / alpha_nb

pois_p0 = sp_poisson.pmf(0, mu_pois).mean()
nb_p0   = (r / (r + mu_nb) ** r).mean()
obs_p0  = (randhie['mdvis'] == 0).mean()

print("=== Zero Rate Comparison ===")
print(f"  Observed zeros:          {obs_p0:.3f}  ({obs_p0*100:.1f}%)")
print(f"  Poisson predicted zeros: {pois_p0:.3f}  ({pois_p0*100:.1f}%)")
print(f"  NegBin predicted zeros:  {nb_p0:.3f}  ({nb_p0*100:.1f}%)")
print(f"  Gap (obs - NB):          {obs_p0 - nb_p0:.3f}")
print("\n  Even NB underpredicts zeros -> zero inflation is present")

# Observed vs model-predicted count distribution
fig, ax = plt.subplots(figsize=(10, 4))
k     = np.arange(0, 20)
obs   = [(randhie['mdvis'] == ki).mean() for ki in k]
p_p   = [sp_poisson.pmf(ki, mu_pois).mean() for ki in k]
p_nb  = [stats.nbinom.pmf(ki, r, r / (r + mu_nb)).mean() for ki in k]

w = 0.28
ax.bar(k - w, obs, w, label='Observed',      color='#1a1d23', alpha=0.85)
ax.bar(k,     p_p, w, label='Poisson',       color='#e85d20', alpha=0.75)
ax.bar(k + w, p_nb,w, label='Neg Binomial',  color='#1e5fa8', alpha=0.75)
ax.set_xlabel('# Doctor Visits')
ax.set_ylabel('Proportion')
ax.set_title('Observed vs Model-Predicted Distribution\n'
             'Both models underpredict zeros in real RAND data')
ax.legend()
ax.set_xlim(-0.5, 15)
plt.tight_layout()
plt.show()

In [ ]:
# Fit Zero-Inflated Negative Binomial
X_count = sm.add_constant(randhie[['lncoins', 'idp', 'physlm', 'disea', 'hlthg', 'hlthf', 'hlthp']])
X_infl  = sm.add_constant(randhie[['lncoins', 'physlm', 'hlthg']])

zinb = ZeroInflatedNegativeBinomialP(
    randhie['mdvis'], X_count, exog_infl=X_infl, p=2
).fit(method='bfgs', maxiter=500, disp=False)

print(zinb.summary())

In [ ]:
# Final model comparison
zinb_p0 = zinb.predict(which='prob-zero').mean()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Zero rate comparison
model_names = ['Observed', 'Poisson', 'Neg Binomial', 'ZINB']
zero_rates  = [obs_p0, pois_p0, nb_p0, zinb_p0]
bar_colors  = ['#1a1d23', '#e85d20', '#1e5fa8', '#1a7a45']

bars = axes[0].bar(model_names, zero_rates, color=bar_colors, edgecolor='white', width=0.5)
for bar, val in zip(bars, zero_rates):
    axes[0].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.003,
                 f'{val:.3f}', ha='center', va='bottom',
                 fontsize=11, fontweight='bold')
axes[0].set_ylabel('Predicted Zero Rate')
axes[0].set_title('Zero Rate: ZINB Matches Observed Best\n(RAND Doctor Visits)')

# AIC comparison
try:
    aic_vals   = [pois_model.aic, nb_model.aic, zinb.aic]
    aic_labels = ['Poisson', 'Neg Binomial', 'ZINB']
    aic_colors = ['#e85d20', '#1e5fa8', '#1a7a45']
    bars2 = axes[1].bar(aic_labels, aic_vals, color=aic_colors, edgecolor='white', width=0.5)
    for bar, val in zip(bars2, aic_vals):
        axes[1].text(bar.get_x() + bar.get_width() / 2,
                     bar.get_height() + 200,
                     f'{val:,.0f}', ha='center', va='bottom',
                     fontsize=10, fontweight='bold')
    axes[1].set_ylabel('AIC (lower = better fit)')
    axes[1].set_title('Model Selection by AIC\n(lower = better)')
except Exception as e:
    axes[1].text(0.5, 0.5, 'See individual AIC values in summaries above',
                 ha='center', va='center', transform=axes[1].transAxes, fontsize=11)

plt.tight_layout()
plt.show()

print("\n=== Final Model Comparison ===")
print(f"  Poisson AIC:   {pois_model.aic:>12,.1f}  (ignores overdispersion)")
print(f"  NegBin AIC:    {nb_model.aic:>12,.1f}  (handles overdispersion)")
try:
    print(f"  ZINB AIC:      {zinb.aic:>12,.1f}  (handles overdispersion + excess zeros)")
    best_aic = min(pois_model.aic, nb_model.aic, zinb.aic)
    best_name = ['Poisson','NegBin','ZINB'][[pois_model.aic,nb_model.aic,zinb.aic].index(best_aic)]
    print(f"\n  Best model: {best_name} (lowest AIC = {best_aic:,.1f})")
except Exception:
    pass

## 🗺️ Part 5 — Choosing the Right Model

| Model | Use when |
|-------|----------|
| **Poisson** | Var ≈ Mean (equidispersion) — rare in practice |
| **Neg Binomial** | Var >> Mean (overdispersion) — most real data |
| **ZIP** | Zero-inflated, equidispersed |
| **ZINB** | Overdispersed AND zero-inflated |

**Quick checks:** Pearson χ²/df >> 1 → NB. Observed zeros >> model zeros → add zero-inflation. Use AIC to compare.

In [ ]:
# ── Exercise workspace ──────────────────────────────────────────────────────
# Task 1: Interpret the lncoins IRR from the NB model in plain English

# Task 2: Load the committees dataset and check for overdispersion
committees = smd.committee.load_pandas().data
print("Committees dataset:", committees.shape)
print(committees.head())
# Fit Poisson on SUBS (subcommittee assignments), check Pearson chi2/df
# YOUR CODE HERE

# Task 3: Predict visits for two hypothetical patients using nb_model
patient_A = pd.DataFrame({'lncoins': [4.6], 'idp': [1], 'physlm': [0],
                           'disea': [0], 'hlthg': [1], 'hlthf': [0], 'hlthp': [0]})
patient_B = pd.DataFrame({'lncoins': [0.0], 'idp': [0], 'physlm': [1],
                           'disea': [5], 'hlthg': [0], 'hlthf': [0], 'hlthp': [1]})
# pred_A = nb_model.predict(sm.add_constant(patient_A, has_constant='add'))
# pred_B = nb_model.predict(sm.add_constant(patient_B, has_constant='add'))
# print(f"Patient A predicted visits: {pred_A.values[0]:.2f}")
# print(f"Patient B predicted visits: {pred_B.values[0]:.2f}")
# print(f"Patient B has {pred_B.values[0]/pred_A.values[0]:.1f}x more predicted visits")
# YOUR CODE HERE

In [ ]:
_NB_NAME_="Count Models — Poisson, Negative Binomial & ZINB"
# @title 🤖 AI Grading — fill in your username and click ▶ Run
# @markdown No setup needed — uses Colab's built-in Gemini connection.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"your GitHub username e.g. jsmith42"}

import textwrap
_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

if "answers" not in globals():
    print("Run the quiz cell above first, then re-run this cell.")
else:
    qa = "\n".join(f"Q{i}: {v or '(no answer)'}"
                    for i,(_,v) in enumerate(answers.items(),1))
    prompt = (f"Grade these quiz answers for the \"{_NB_TITLE}\" notebook "
              f"(Data Pattern Recognition for the Rest of Us, based on ISLP).\n\n"
              f"{qa}\n\n"
              f"For each answer say CORRECT, PARTIAL, or INCORRECT and one sentence why. "
              f"Accept any reasonable paraphrase as correct. "
              f"End with an overall grade (Excellent/Good/Needs Review/Incomplete) "
              f"and one sentence on what to review if they struggled.")
    try:
        import google.generativeai as genai
        response = genai.GenerativeModel("gemini-2.0-flash").generate_content(prompt)
        print("\n" + "\u2550"*56)
        print(f"  \U0001f916 Feedback \u2014 {_NB_TITLE}")
        if GITHUB_USERNAME.strip():
            print(f"  Student: @{GITHUB_USERNAME.strip()}")
        print("\u2550"*56)
        for line in response.text.strip().split("\n"):
            if line.strip():
                for w in textwrap.wrap(line.strip(), 54):
                    print(f"  {w}")
            else:
                print()
        print("\u2550"*56)
    except Exception as e:
        print(f"Gemini error: {e}")
    print("\n  \U0001f4be File \u2192 Save a copy in GitHub \u2192 your fork")


In [ ]:
# @title 📝 Quiz — Count Models — Poisson, Negative Binomial & ZINB
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** What does the Poisson assumption Var(Y)=E(Y) mean, and how do you test if it is violated?
# @markdown **Q2:** How do you interpret an Incidence Rate Ratio (IRR) of 1.45 for a binary predictor like 'smoker'?
# @markdown **Q3:** What is the difference between structural zeros and sampling zeros in count data?
# @markdown **Q4:** When would you choose ZINB over Negative Binomial?
# @markdown **Q5:** Why are Poisson standard errors too small when data is overdispersed?

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
answered = sum(1 for v in answers.values() if str(v).strip())
print(f"  {answered}/5 answered — run the AI grading cell below")

## 📚 Further Reading

| Resource | What it covers |
|----------|---------------|
| [ISLP Ch 4.6](https://intro-stat-learning.github.io/ISLP/) | Poisson regression |
| [statsmodels GLM docs](https://www.statsmodels.org/stable/glm.html) | Poisson & NB in Python |
| [Logistic Regression notebook](logistic-regression.ipynb) | Binary outcome — the classification analogue |

---
*Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*